In [75]:
import cv2
import numpy as np
import random

agent_x, agent_y = 0, 350
gamma = 0.9

def create_env():
    obstacles=[
        (1,1),
        (4,1),
        (2,3),
        (5,3),
        (5,4),
        (5,5)
    ]
    
    canvas = np.zeros((400,400,3),dtype = np.uint8)
    
    for i in range(0,401,50):
        cv2.line(canvas,(0,i),(400,i),(255,255,255),1)
        cv2.line(canvas,(i,0),(i,400),(255,255,255),1)
    
    #cv2.rectangle(canvas,(agent_x+1,agent_y+1),(agent_x+50-1,agent_y+50-1),(255,0,0),-1)
    cv2.rectangle(canvas,(350,0),(400,50),(255,0,255),-1)

    for i in obstacles:
        x1 =[i[1]][0]*50
        y1 =[i[0]][0]*50
        x2 = x1+50
        y2 = y1+50
    
        cv2.rectangle(canvas,(x1,y1),(x2,y2),(0,255,255),-1)

    return canvas

canvas = create_env()

actions = {0:[0, -50], 1:[0, 50], 2:[-50, 0], 3:[50, 0]} # up,down,left,right

def env(state, action, agent_x, agent_y, b ,g, r):
    cv2.rectangle(state,(agent_x+1,agent_y+1),(agent_x+50-1,agent_y+50-1),(b ,g, r),-1)
    next_agent_x = actions[action][0] + agent_x
    next_agent_y = actions[action][1] + agent_y
    
    b, g, r = state[(next_agent_y+1+next_agent_y+50-1)//2,(next_agent_x+1+next_agent_x+50-1)//2,:]
    b, g, r = b.item(), g.item(), r.item()
    
    cv2.rectangle(state,(next_agent_x+1,next_agent_y+1),(next_agent_x+50-1,next_agent_y+50-1),(255,0,0),-1)
    reward = 0
    if b==0 and g==255 and r==255:
        reward = -100
    if b==255 and g==0 and r==255:
        reward = 100
    return state, next_agent_x, next_agent_y, b ,g, r, reward

def valid_actions(current_state):
    actions_list = [0, 1, 2, 3]
    if current_state==0:
        actions_list = [1, 3]
    elif current_state==56:
        actions_list = [0, 3]
    elif current_state==63:
        actions_list = [0, 2]
    elif 1<=current_state<=6:
        actions_list = [1, 2, 3]
    elif 57<=current_state<=62:
        actions_list = [0, 2, 3]
    elif current_state%8==0 and current_state!=0 and current_state!=56:
        actions_list = [0, 1, 3]
    elif (current_state+1)%8==0 and current_state!=7 and current_state!=63:
        actions_list = [0, 1, 2]
    return actions_list

Q = np.zeros((64, 4))
b ,g, r = 0,0,0
invalid = [7,9,19,33,43,44,45]

valid_positions = [i for i in range(64) if i not in invalid]

for _ in range(10000):
    current_state = np.arange(64).reshape(8,8)[agent_y//50, agent_x//50]
    actions_list = valid_actions(current_state)

    current_action = random.choice(actions_list)
    # epsilon = 0.8 # 80% chance to exploit (use Q-table), 20% chance to explore (random)

    # if random.uniform(0, 1) > epsilon:
    #     # EXPLORE: Take a random action
    #     current_action = random.choice(actions_list)
    # else:
    #     # EXPLOIT: Look at the Q-table and pick the best valid action
    #     # We create a dictionary of only the valid actions and their current Q-values
    #     valid_q_values = {a: Q[current_state, a] for a in actions_list}
        
    #     # Pick the action that has the highest Q-value
    #     current_action = max(valid_q_values, key=valid_q_values.get)

    canvas, agent_x, agent_y, b ,g, r, reward = env(canvas, current_action, agent_x, agent_y, b ,g, r)
    
    state_ = np.arange(64).reshape(8,8)[agent_y//50, agent_x//50]

    Q_ = Q[state_][np.argmax(Q[state_])]

    if state_== 7:
        canvas = create_env()
        xy = random.choice(valid_positions)
        row = xy // 8
        col = xy % 8
        agent_x = col * 50
        agent_y = row * 50

        #agent_x, agent_y = 0, 350
        b, g, r = 0,0,0
        Q_ = 0
    elif state_ in [9,19,33,43,44,45]:
        canvas = create_env()
        xy = random.choice(valid_positions)
        row = xy // 8
        col = xy % 8
        agent_x = col * 50
        agent_y = row * 50

        #agent_x, agent_y = 0, 350
        b, g, r = 0,0,0
        Q_ = 0

    Q[current_state,current_action] = reward + gamma * Q_

#     cv2.imshow("Video", canvas)

#     # Press 'q' to quit
#     if cv2.waitKey(250) & 0xFF == ord('q'):
#         break
# cv2.destroyAllWindows()

print(Q)

[[   0.           43.046721      0.           53.1441    ]
 [   0.         -100.           47.82969      59.049     ]
 [   0.           53.1441       53.1441       65.61      ]
 [   0.           59.049        59.049        72.9       ]
 [   0.           65.61         65.61         81.        ]
 [   0.           72.9          72.9          90.        ]
 [   0.           81.           81.          100.        ]
 [   0.            0.            0.            0.        ]
 [  47.82969      38.7420489     0.         -100.        ]
 [   0.            0.            0.            0.        ]
 [  59.049        47.82969    -100.           59.049     ]
 [  65.61       -100.           53.1441       65.61      ]
 [  72.9          59.049        59.049        72.9       ]
 [  81.           65.61         65.61         81.        ]
 [  90.           72.9          72.9          90.        ]
 [ 100.           81.           81.            0.        ]
 [  43.046721     34.86784401    0.           43.046721 

In [94]:
flag = 0
for _ in range(10):
    canvas = create_env()
    xy = random.choice(valid_positions)
    row = xy // 8
    col = xy % 8
    agent_x = col * 50
    agent_y = row * 50
    #agent_x, agent_y = 200, 200
    reward = 0
    b, g, r = 0,0,0
    while 1:
        current_state = np.arange(64).reshape(8,8)[agent_y//50, agent_x//50]
        if current_state==7:
            break
        current_action = np.argmax(Q[current_state])
        
        canvas, agent_x, agent_y, b ,g, r, reward = env(canvas, current_action, agent_x, agent_y, b ,g, r)

        cv2.imshow("Video", canvas)

        # Press 'q' to quit
        if cv2.waitKey(250) & 0xFF == ord('q'):
            flag=1
            break
    if flag==1:
        break
cv2.destroyAllWindows()

64